<a href="https://colab.research.google.com/github/deartoms/python/blob/main/selenium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from selenium import webdriver #브라우저를 실행하고 제어
from selenium.webdriver.chrome.service import Service # 크롬드라이버 실행 파일의 경로와 실행 방식을 관리
from selenium.webdriver.chrome.options import Options # 브라우저 옵션(설정)을 지정
from selenium.webdriver.common.by import By #HTML 요소를 찾는 방법을 지정 By.ID / By.CSS_SELECTOR 등
from selenium.webdriver.support.ui import WebDriverWait # 특정 조건이 만족될 때까지 대기
from selenium.webdriver.support import expected_conditions as EC #요소가 HTML에 존재할 때까지 대기
from webdriver_manager.chrome import ChromeDriverManager # 현재 크롬 버전에 맞는 크롬드라이버를 자동으로 다운로드
from bs4 import BeautifulSoup # HTML 문자열을 파싱해서 원하는 태그와
import pandas as pd
import time #지정한 초만큼 실행을 멈추고 대기

크롬 드라이버 실행

In [ ]:
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)# detach=True : 코드 실행이 끝나도 브라우저가 자동으로 닫히지 않습니다
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])# 크롬 실행 시 출력되는 불필요한 로그 메시지를 숨깁니다

service = Service(ChromeDriverManager().install())# 알맞은 비전의 크롬드라이버를 자동 설치하고 경로를 반환
# 반환된 경로로 드라이버 실행 서비스 객체를 만듭니다
driver = webdriver.Chrome(service=service, options=chrome_options) # 실제 크롬 브라우저를 실행
print("드라이버 실행 완료")# 이후 모든 조작은 이 driver 객체로 합니다

드라이버 실행 완료


페이지 접속/ 상품 링크 10개 수집

In [ ]:
main_url = "https://www.oliveyoung.co.kr/store/main/getBestList.do?dispCatNo=900000100100001&fltDispCatNo=10000010001&pageIdx=1&rowsPerPage=8"

driver.get(main_url)# 브라우저가 해당 URL로 이동

wait = WebDriverWait(driver, 300) # 봇 접근 시 캡차를 띄우므로 수동으로 풀 수 있게 최대 300초(5분) 동안 대기

wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.best-area"))) # 요소가 페이지에 나타날 때까지 기다려라
time.sleep(2) # 요소 감지 후 2초 추가 대기,페이지 내 다른 요소들까지 완전히 렌더링될 시간

soup = BeautifulSoup(driver.page_source, "html.parser") #현재 브라우저에 렌더링된 HTML 전체를 문자열로 가져와서 파싱하여 soup 객체를 만듬
items = soup.select("#Container > div.best-area > div.TabsConts.on > ul > li") #조건에 맞는 요소를 찾아 리스트로 반환
sub_url = [item.select_one("div.prd_info > a.prd_thumb")["href"] for item in items[:10]] # 리스트 컴프리헨션으로 각 상품 카드에서 링크(href)를 뽑아 sub_url 리스트를 만듭니다

print(f"수집된 링크: {len(sub_url)}개")

for i, url in enumerate(sub_url, 1):
    print(f"{i}, {url}")

수집된 링크: 10개
1, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000226498&dispCatNo=90000010009&trackingCd=Best_Sellingbest&t_page=랭킹&t_click=판매랭킹_스킨케어_상품상세&t_number=1
2, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000206698&dispCatNo=90000010009&trackingCd=Best_Sellingbest&t_page=랭킹&t_click=판매랭킹_스킨케어_상품상세&t_number=2
3, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000247193&dispCatNo=90000010009&trackingCd=Best_Sellingbest&t_page=랭킹&t_click=판매랭킹_스킨케어_상품상세&t_number=3
4, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000226515&dispCatNo=90000010009&trackingCd=Best_Sellingbest&t_page=랭킹&t_click=판매랭킹_스킨케어_상품상세&t_number=4
5, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000247770&dispCatNo=90000010009&trackingCd=Best_Sellingbest&t_page=랭킹&t_click=판매랭킹_스킨케어_상품상세&t_number=5
6, https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000206904&dispCatNo=9000

상품 상세 페이지 접속 &

In [ ]:
result = [] # 수집 결과를 담을 빈 리스트

for i, url in enumerate(sub_url, 1):
    print(f"[{i}/10] 크롤링 중...", end=" ")

    driver.get(url) # 브라우저가 상품 상세 페이지로 이동
    time.sleep(3) #JavaScript가 상품 정보를 렌더링할 시간을 확보
    soup = BeautifulSoup(driver.page_source, "html.parser") # 상세페이지 html을 파싱

    # 브랜드명
    try:
        brand = soup.select_one('[class*="btn-brand"]').get_text(strip=True)
    except:
        brand = "없음"


    # 상품명
    try:
        name = soup.select_one('[class*="GoodsDetailInfo_title"]').get_text(strip=True)
    except:
        name = "없음"

    # 카테고리
    try:
        breacrumb = soup.select_one('[class*="Breadcrumb_breadcrumb-inner"]')
        links = breacrumb.select('a[role="link"]') # 리스트 컴프리헨션 + join() : ["스킨케어","크림","크림"] ->
        "스킨케어 > 크림 > 크림"
        category = ">".join([a.get_text(strip=True) for a in links])
    except:
        category = "없음"
    # 정가
    try:
        price = soup.select_one('[data-qa-name="text-product-original-price"] span:first-child').get_text(strip=True) + "원"
    except:
        price = "없음"

    # 할인가
    try:
        discount = soup.select_one('[data-qa-name="text-product-discount-price"] span:first-child').get_text(strip=True) + "원"
    except:
        discount = "없음"

    # 평점
    try:
        rating_tag = soup.select_one('span.rating') # select_one() : 조건에 맞는 요소를 하나만 반환
        for blind in rating_tag.select('.oyblind'): # select() : 조건에 맞는 요소를 모두 찾아 리스트로 반환
            blind.decompose() # decompose()로 제거
        rating = rating_tag.get_text(strip=True)
    except:
        rating = '없음'

    # 리뷰건수
    try:
        review_count = soup.select_one('div[class*="review-count"] button span').get_text(strip=True) + "건"
    except:
        review_count = "없음"

    result.append([i, brand, name, category, price, discount, rating, review_count])

    print(f"{brand} / {name[:25]} / 평점 {rating} / 리뷰 {review_count}")

driver.quit() # 모든 수집이 끝나면 브라우저를 종료
print(f"\n 수집 완료 - 총 {len(result)}개")

[1/10] 크롤링 중... 메디큐브 / [3월 올영픽/1등미백앰플] 메디큐브 PDRN / 평점 4.7 / 리뷰 20,262건
[2/10] 크롤링 중... 빌리프 / [3월올영픽/생기충전][대용량] 빌리프X바나나 / 평점 4.9 / 리뷰 2,782건
[3/10] 크롤링 중... 토리든 / [3월올영픽/1위세럼]토리든 다이브인 저분자  / 평점 4.8 / 리뷰 72,479건
[4/10] 크롤링 중... 브링그린 / [3월올영PICK|안경만두|ALD1] 브링그린 / 평점 4.7 / 리뷰 16,246건
[5/10] 크롤링 중... 아누아 / [단독기획] 아누아 피디알엔 히알루론산 캡슐  / 평점 4.8 / 리뷰 18,868건
[6/10] 크롤링 중... 라로슈포제 / 라로슈포제 시카플라스트 밤 B5+ 100ml  / 평점 4.8 / 리뷰 22,713건
[7/10] 크롤링 중... 에스네이처 / [리뷰이벤트/1+1/모공 수분천재크림] 에스네 / 평점 4.8 / 리뷰 32,894건
[8/10] 크롤링 중... 달바 / [NO.1 미스트세럼] 달바 퍼스트 스프레이  / 평점 4.9 / 리뷰 44,615건
[9/10] 크롤링 중... 메디큐브 / [3월 올영픽/여배우광채크림]메디큐브 PDRN / 평점 4.8 / 리뷰 140건
[10/10] 크롤링 중... 메디힐 / [3월 올영픽/1+1] 메디힐 마데카소사이드  / 평점 4.7 / 리뷰 38,504건

 수집 완료 - 총 10개


데이터프레임 확인

In [ ]:
columns = ["순위", "브랜드","상품명","카테고리","정가","할인가","평점", "리뷰건수"]
df = pd.DataFrame(result, columns=columns)
df

,순위,브랜드,상품명,카테고리,정가,할인가,평점,리뷰건수
0,1,메디큐브,[3월 올영픽/1등미백앰플] 메디큐브 PDRN 핑크 앰플 30ml 기획 (본품+리필...,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"46,000원","24,600원",4.7,"20,262건"
1,2,빌리프,[3월올영픽/생기충전][대용량] 빌리프X바나나킥 비타C 토닝 세럼 50ml 기획 (...,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"44,000원","33,000원",4.9,"2,782건"
2,3,토리든,[3월올영픽/1위세럼]토리든 다이브인 저분자 히알루론산 세럼 50ml 한정 리필 기획,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"36,000원","27,000원",4.8,"72,479건"
3,4,브링그린,[3월올영PICK|안경만두|ALD1] 브링그린 징크테카 트러블세럼 (콜라보/대용량/기획),스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"56,100원","29,800원",4.7,"16,246건"
4,5,아누아,[단독기획] 아누아 피디알엔 히알루론산 캡슐 100 세럼 30ml 더블 기획,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"57,500원","29,900원",4.8,"18,868건"
5,6,라로슈포제,라로슈포제 시카플라스트 밤 B5+ 100ml 기획 (+3ml 추가증정),스킨케어>크림>크림,"41,000원","36,900원",4.8,"22,713건"
6,7,에스네이처,[리뷰이벤트/1+1/모공 수분천재크림] 에스네이처 아쿠아 스쿠알란 수분크림 60ml...,스킨케어>크림>크림,"43,000원","22,900원",4.8,"32,894건"
7,8,달바,[NO.1 미스트세럼] 달바 퍼스트 스프레이 세럼 100ml 2개 기획,스킨케어>미스트/오일>미스트/픽서,"59,800원","32,900원",4.9,"44,615건"
8,9,메디큐브,[3월 올영픽/여배우광채크림]메디큐브 PDRN 핑크 콜라겐 캡슐크림 55g 기획 (...,스킨케어>크림>크림,"33,300원","23,800원",4.8,140건
9,10,메디힐,[3월 올영픽/1+1] 메디힐 마데카소사이드 흔적 리페어 세럼 40+40ml 더블 기획,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,"36,900원","22,900원",4.7,"38,504건"


엑셀 파일 저장

In [ ]:
df.to_excel("올리브영_랭킹TOP10.xlsx", index=False)
print("엑셀 저장 완료 - 올리브영_랭킹TOP10.xlsx")

엑셀 저장 완료 - 올리브영_랭킹TOP10.xlsx
